# PDB RCSB Derived Metadata Pipeline

Fetches per-entry metadata from the RCSB GraphQL API for all released PDB entries
(~226 K), writing one NDJSON file per entity type to the Lakehouse bronze layer.
Previous versions are archived automatically when content changes.

## Entity types fetched
| Entity | Output file | Contents |
|--------|-------------|----------|
| `entries` | `entries.ndjson` | Core metadata: method, resolution, dates, organism, DOI |
| `validation` | `validation.ndjson` | wwPDB validation scores (clashscore, Ramachandran, RMSZ) |
| `taxonomy` | `taxonomy.ndjson` | Full NCBI taxonomy lineage per polymer entity |
| `ligands` | `ligands.ndjson` | Bound ligand chemical components |
| `citations` | `citations.ndjson` | Primary and related citations, DOIs, PubMed IDs |
| `pfam` | `pfam.ndjson` | Pfam domain annotations per polymer entity |
| `sequence_clusters` | `sequence_clusters.ndjson` | RCSB sequence cluster membership |

## Output paths
| Object | S3 path |
|--------|--------|
| Current file | `s3://{LAKEHOUSE_BUCKET}/{LAKEHOUSE_KEY_PREFIX}/derived_data/rcsb/{entity_type}.ndjson` |
| Archived (on change) | `s3://{LAKEHOUSE_BUCKET}/{LAKEHOUSE_KEY_PREFIX}/derived_data/archive/{YYYY-MM-DD}/rcsb/{entity_type}.ndjson` |

In [ ]:
"""Imports."""

from cdm_data_loaders.rcsb_metadata.run import run_rcsb_metadata
from cdm_data_loaders.rcsb_metadata.settings import RcsbMetadataSettings

In [ ]:
"""Configure the RCSB metadata pipeline.

Set DRY_RUN = True to log what would happen without making any API calls
or S3 uploads.
"""

# S3 bucket for the Lakehouse bronze layer
# format: bucket name (no s3:// scheme)
LAKEHOUSE_BUCKET = "cdm-lake"

# S3 key prefix for PDB datasets
# format: S3 key prefix
LAKEHOUSE_KEY_PREFIX = "tenant-general-warehouse/kbase/datasets/pdb"

# Number of PDB IDs per GraphQL request (reduce if you hit rate limits)
BATCH_SIZE = 1000

# Limit to first N entries (None = process all ~226 K; set e.g. 100 for testing)
LIMIT: int | None = 100

# Set to True to skip all API calls and uploads (log-only mode)
DRY_RUN = False

settings = RcsbMetadataSettings(
    lakehouse_bucket=LAKEHOUSE_BUCKET,
    lakehouse_key_prefix=LAKEHOUSE_KEY_PREFIX,
    rcsb_batch_size=BATCH_SIZE,
    limit=LIMIT,
    dry_run=DRY_RUN,
)
print(settings.model_dump())

In [ ]:
"""Configure S3 credentials (use for local testing against the MinIO test container).

Set PROVIDE_CREDENTIALS = True to override the default credential chain and
point at a local MinIO instance instead of AWS.  Leave False when running in
an environment that already has credentials (IAM role, environment variables,
~/.aws/credentials, etc.).
"""

from cdm_data_loaders.utils.s3 import get_s3_client, reset_s3_client

PROVIDE_CREDENTIALS = False  # Set to True to use the credentials below instead of environment credentials
if PROVIDE_CREDENTIALS:
    reset_s3_client()  # Clear any existing client to ensure new credentials are used
    get_s3_client(
        args={
            "endpoint_url": "http://localhost:9000",
            "aws_access_key_id": "minioadmin",
            "aws_secret_access_key": "minioadmin",
        }
    )
    print("Using local MinIO credentials")
else:
    print("Using environment / IAM credentials")

In [ ]:
"""Run the RCSB metadata pipeline."""

result = run_rcsb_metadata(settings)

print(f"Total PDB entries : {result.total_entries}")
print(f"Dry run           : {result.dry_run}")
print(f"Descriptor        : {result.descriptor_key or '(not written)'}")
print()
print(f"{'Entity type':<25} {'Status':<25} {'Records':>8} {'Archive key'}")
print("-" * 80)
for entity_type, er in result.entity_results.items():
    print(f"{entity_type:<25} {er.upload_status:<25} {er.records_written:>8}  {er.archive_key or ''}")